# Universal Transcript Data Analysis
## Quantitative Analysis Framework for Any Conversation Transcript

This notebook provides a reusable framework for analyzing conversation transcripts using pure data science methods. No AI or NLP dependencies - just statistical analysis, pattern recognition, and data visualization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
from datetime import datetime, timedelta
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Universal Transcript Analysis Framework Loaded")

Universal Transcript Analysis Framework Loaded


## 1. Universal SRT Parser Class

In [3]:
class TranscriptParser:
    """Universal parser for SRT subtitle files"""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
    
    def time_to_seconds(self, time_str):
        """Convert timestamp to seconds"""
        time_parts = time_str.replace('.', ':').split(':')
        if len(time_parts) == 4:
            hours, minutes, seconds, milliseconds = map(int, time_parts)
            return hours * 3600 + minutes * 60 + seconds + milliseconds / 1000
        return 0
    
    def extract_speaker(self, text):
        """Extract speaker from text using common patterns"""
        patterns = [
            r'^([^:()]+)\s*\([^)]+\)\s*:', # Name (Role):
            r'^([^:]+):', # Name:
            r'^\[([^\]]+)\]', # [Name]
            r'^([A-Z][a-z]+)\s*-', # Name -
        ]
        
        for pattern in patterns:
            match = re.match(pattern, text.strip())
            if match:
                speaker = match.group(1).strip()
                message = text[match.end():].strip()
                return speaker, message
        
        return "Unknown", text
    
    def parse(self):
        """Parse SRT file and return DataFrame"""
        with open(self.file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        
        blocks = content.strip().split('\n\n')
        conversations = []
        
        for block in blocks:
            lines = block.strip().split('\n')
            if len(lines) >= 3:
                try:
                    subtitle_num = int(lines[0])
                    timestamp_line = lines[1]
                    
                    # Extract timestamps - fixed regex pattern
                    time_match = re.match(r'(\d{2}:\d{2}:\d{2}[,.]\d{3}) --> (\d{2}:\d{2}:\d{2}[,.]\d{3})', timestamp_line)
                    if time_match:
                        start_time = time_match.group(1).replace(',', '.')
                        end_time = time_match.group(2).replace(',', '.')
                        
                        text = ' '.join(lines[2:])
                        speaker, message = self.extract_speaker(text)
                        
                        conversations.append({
                            'subtitle_num': subtitle_num,
                            'start_time': start_time,
                            'end_time': end_time,
                            'speaker': speaker,
                            'message': message,
                            'full_text': text
                        })
                except (ValueError, IndexError):
                    continue
        
        self.df = pd.DataFrame(conversations)
        if not self.df.empty:
            self.df['start_seconds'] = self.df['start_time'].apply(self.time_to_seconds)
            self.df['end_seconds'] = self.df['end_time'].apply(self.time_to_seconds)
            self.df['duration'] = self.df['end_seconds'] - self.df['start_seconds']
            self.df['word_count'] = self.df['message'].apply(lambda x: len(str(x).split()))
            self.df['char_count'] = self.df['message'].apply(lambda x: len(str(x)))
            self.df['question_marks'] = self.df['message'].apply(lambda x: str(x).count('?'))
            self.df['exclamation_marks'] = self.df['message'].apply(lambda x: str(x).count('!'))
            
        return self.df

# Initialize parser
parser = TranscriptParser('transcript.srt')
df = parser.parse()

print(f"Parsed {len(df)} conversation segments")
print(f"Unique speakers: {df['speaker'].nunique()}")
print(f"Total duration: {df['end_seconds'].max():.1f} seconds ({df['end_seconds'].max()/60:.1f} minutes)")

df.head()

Parsed 67 conversation segments
Unique speakers: 3
Total duration: 1520.0 seconds (25.3 minutes)


,subtitle_num,start_time,end_time,speaker,message,full_text,start_seconds,end_seconds,duration,word_count,char_count,question_marks,exclamation_marks
0,1,00:00:00.000,00:00:18.000,Arjun,"Hi Meera, good afternoon. Can you hear me clea...","Arjun (Pre-Sales): Hi Meera, good afternoon. C...",0.0,18.0,18.0,9,50,1,0
1,2,00:00:18.500,00:00:28.000,Meera,"Yes, loud and clear. Thanks for joining. I kno...","Meera (Client): Yes, loud and clear. Thanks fo...",18.5,28.0,9.5,14,79,0,0
2,3,00:00:28.500,00:00:42.000,Arjun,"Absolutely. I always say, the hardest part of ...","Arjun: Absolutely. I always say, the hardest p...",28.5,42.0,13.5,24,130,1,0
3,4,00:00:42.500,00:00:57.000,Meera,Busy but good. We’re preparing for a board upd...,Meera: Busy but good. We’re preparing for a bo...,42.5,57.0,14.5,23,127,0,0
4,5,00:00:57.500,00:01:13.000,Arjun,Perfect. Then let’s make this session useful. ...,Arjun: Perfect. Then let’s make this session u...,57.5,73.0,15.5,38,216,1,0


## 2. Speaker Analysis Class

In [4]:
class SpeakerAnalyzer:
    """Analyze speaker patterns and behavior"""
    
    def __init__(self, df):
        self.df = df.copy()
    
    def calculate_speaker_stats(self):
        """Calculate comprehensive speaker statistics"""
        stats = self.df.groupby('speaker').agg({
            'subtitle_num': 'count',
            'duration': ['sum', 'mean', 'std'],
            'word_count': ['sum', 'mean', 'std'],
            'char_count': ['sum', 'mean'],
            'question_marks': 'sum',
            'exclamation_marks': 'sum'
        }).round(2)
        
        stats.columns = ['turns', 'total_duration', 'avg_duration', 'std_duration',
                        'total_words', 'avg_words', 'std_words',
                        'total_chars', 'avg_chars', 'questions', 'exclamations']
        
        # Calculate additional metrics
        total_words = stats['total_words'].sum()
        total_duration = stats['total_duration'].sum()
        
        stats['word_percentage'] = (stats['total_words'] / total_words * 100).round(1)
        stats['time_percentage'] = (stats['total_duration'] / total_duration * 100).round(1)
        stats['words_per_second'] = (stats['total_words'] / stats['total_duration']).round(2)
        
        return stats
    
    def analyze_speaking_patterns(self):
        """Analyze when each speaker talks"""
        patterns = []
        
        for speaker in self.df['speaker'].unique():
            speaker_df = self.df[self.df['speaker'] == speaker]
            
            # Calculate gaps between speaking turns
            gaps = speaker_df['start_seconds'].diff().dropna()
            
            patterns.append({
                'speaker': speaker,
                'first_appearance': speaker_df['start_seconds'].min(),
                'last_appearance': speaker_df['end_seconds'].max(),
                'avg_gap_between_turns': gaps.mean() if len(gaps) > 0 else 0,
                'max_gap_between_turns': gaps.max() if len(gaps) > 0 else 0,
                'speaking_span': speaker_df['end_seconds'].max() - speaker_df['start_seconds'].min()
            })
        
        return pd.DataFrame(patterns)
    
    def calculate_dominance_score(self):
        """Calculate conversation dominance metrics"""
        stats = self.calculate_speaker_stats()
        
        # Normalize metrics to 0-1 scale
        metrics = ['word_percentage', 'time_percentage', 'turns']
        dominance_scores = {}
        
        for speaker in stats.index:
            score = (
                stats.loc[speaker, 'word_percentage'] * 0.4 +
                stats.loc[speaker, 'time_percentage'] * 0.4 +
                (stats.loc[speaker, 'turns'] / stats['turns'].max() * 100) * 0.2
            )
            dominance_scores[speaker] = round(score, 1)
        
        return dominance_scores

# Analyze speakers
analyzer = SpeakerAnalyzer(df)
speaker_stats = analyzer.calculate_speaker_stats()
speaking_patterns = analyzer.analyze_speaking_patterns()
dominance_scores = analyzer.calculate_dominance_score()

print("SPEAKER STATISTICS:")
display(speaker_stats)

print("\nDOMINANCE SCORES:")
for speaker, score in dominance_scores.items():
    print(f"{speaker}: {score}%")

SPEAKER STATISTICS:


,turns,total_duration,avg_duration,std_duration,total_words,avg_words,std_words,total_chars,avg_chars,questions,exclamations,word_percentage,time_percentage,words_per_second
speaker,,,,,,,,,,,,,,
Arjun,33,787.0,23.85,3.76,692,20.97,6.74,4382,132.79,29,0,57.9,52.9,0.88
Meera,33,680.5,20.62,4.94,503,15.24,10.24,3225,97.73,0,0,42.1,45.8,0.74
Small talk as both smile and exchange pleasantries before leaving the call.,1,19.5,19.50,NaN,0,0.00,NaN,0,0.00,0,0,0.0,1.3,0.00



DOMINANCE SCORES:
Arjun: 64.3%
Meera: 55.2%
Small talk as both smile and exchange pleasantries before leaving the call.: 1.1%


## 3. Conversation Flow Analysis

In [5]:
class ConversationFlowAnalyzer:
    """Analyze conversation flow and dynamics"""
    
    def __init__(self, df):
        self.df = df.copy()
    
    def calculate_turn_taking_metrics(self):
        """Calculate turn-taking patterns"""
        # Calculate speaker changes
        self.df['speaker_change'] = (self.df['speaker'] != self.df['speaker'].shift()).astype(int)
        
        # Calculate response times (gaps between speakers)
        self.df['response_time'] = self.df['start_seconds'] - self.df['end_seconds'].shift()
        self.df['response_time'] = self.df['response_time'].clip(lower=0)  # Remove negative values
        
        # Calculate interruptions (negative response time indicates overlap)
        self.df['interruption'] = (self.df['start_seconds'] < self.df['end_seconds'].shift()).astype(int)
        
        metrics = {
            'total_turns': len(self.df),
            'speaker_changes': self.df['speaker_change'].sum(),
            'avg_response_time': self.df['response_time'].mean(),
            'median_response_time': self.df['response_time'].median(),
            'total_interruptions': self.df['interruption'].sum(),
            'interruption_rate': (self.df['interruption'].sum() / len(self.df)) * 100
        }
        
        return metrics
    
    def analyze_conversation_phases(self, num_phases=5):
        """Divide conversation into phases and analyze each"""
        total_duration = self.df['end_seconds'].max()
        phase_duration = total_duration / num_phases
        
        phases = []
        
        for i in range(num_phases):
            start_time = i * phase_duration
            end_time = (i + 1) * phase_duration
            
            phase_df = self.df[
                (self.df['start_seconds'] >= start_time) & 
                (self.df['start_seconds'] < end_time)
            ]
            
            if not phase_df.empty:
                phase_stats = {
                    'phase': i + 1,
                    'start_time': start_time,
                    'end_time': end_time,
                    'turns': len(phase_df),
                    'total_words': phase_df['word_count'].sum(),
                    'avg_words_per_turn': phase_df['word_count'].mean(),
                    'questions': phase_df['question_marks'].sum(),
                    'exclamations': phase_df['exclamation_marks'].sum(),
                    'unique_speakers': phase_df['speaker'].nunique(),
                    'dominant_speaker': phase_df.groupby('speaker')['word_count'].sum().idxmax()
                }
                phases.append(phase_stats)
        
        return pd.DataFrame(phases)
    
    def calculate_engagement_metrics(self):
        """Calculate overall engagement quality"""
        turn_metrics = self.calculate_turn_taking_metrics()
        
        # Calculate balance (how evenly distributed is the conversation)
        speaker_word_counts = self.df.groupby('speaker')['word_count'].sum()
        
        if len(speaker_word_counts) > 1:
            # Calculate Gini coefficient for word distribution
            word_values = sorted(speaker_word_counts.values)
            n = len(word_values)
            cumsum = np.cumsum(word_values)
            gini = (n + 1 - 2 * sum((n + 1 - i) * y for i, y in enumerate(cumsum))) / (n * sum(word_values))
            balance_score = (1 - gini) * 100  # Convert to balance score (higher = more balanced)
        else:
            balance_score = 0
        
        engagement = {
            'conversation_balance': round(balance_score, 1),
            'interaction_density': turn_metrics['speaker_changes'] / (self.df['end_seconds'].max() / 60),  # changes per minute
            'avg_turn_length': self.df['word_count'].mean(),
            'question_frequency': self.df['question_marks'].sum() / len(self.df),
            'responsiveness': 1 / turn_metrics['avg_response_time'] if turn_metrics['avg_response_time'] > 0 else 0
        }
        
        return engagement

# Analyze conversation flow
flow_analyzer = ConversationFlowAnalyzer(df)
turn_metrics = flow_analyzer.calculate_turn_taking_metrics()
phases = flow_analyzer.analyze_conversation_phases()
engagement = flow_analyzer.calculate_engagement_metrics()

print("TURN-TAKING METRICS:")
for key, value in turn_metrics.items():
    print(f"{key}: {value:.2f}" if isinstance(value, (int, float)) else f"{key}: {value}")

print("\nENGAGEMENT METRICS:")
for key, value in engagement.items():
    print(f"{key}: {value:.2f}")

print("\nCONVERSATION PHASES:")
display(phases)

TURN-TAKING METRICS:
total_turns: 67.00
speaker_changes: 67
avg_response_time: 0.50
median_response_time: 0.50
total_interruptions: 0
interruption_rate: 0.00

ENGAGEMENT METRICS:
conversation_balance: 317.40
interaction_density: 2.64
avg_turn_length: 17.84
question_frequency: 0.43
responsiveness: 2.00

CONVERSATION PHASES:


,phase,start_time,end_time,turns,total_words,avg_words_per_turn,questions,exclamations,unique_speakers,dominant_speaker
0,1,0.0,304.0,16,362,22.625000,9,0,2,Meera
1,2,304.0,608.0,13,264,20.307692,9,0,2,Arjun
2,3,608.0,912.0,12,219,18.250000,5,0,2,Arjun
3,4,912.0,1216.0,14,218,15.571429,4,0,2,Arjun
4,5,1216.0,1520.0,12,132,11.000000,2,0,3,Arjun


## 4. Visualization Dashboard

In [6]:
class VisualizationDashboard:
    """Create comprehensive visualization dashboard"""
    
    def __init__(self, df, speaker_stats, phases, engagement):
        self.df = df
        self.speaker_stats = speaker_stats
        self.phases = phases
        self.engagement = engagement
    
    def create_speaker_distribution_chart(self):
        """Speaker contribution pie chart"""
        fig = px.pie(
            values=self.speaker_stats['total_words'],
            names=self.speaker_stats.index,
            title="Word Distribution by Speaker",
            hover_data=['total_duration']
        )
        return fig
    
    def create_timeline_chart(self):
        """Conversation timeline with speaker changes"""
        fig = px.scatter(
            self.df,
            x='start_seconds',
            y='speaker',
            size='word_count',
            color='speaker',
            hover_data=['duration', 'word_count'],
            title="Conversation Timeline"
        )
        
        fig.update_xaxes(title="Time (seconds)")
        fig.update_yaxes(title="Speaker")
        return fig
    
    def create_engagement_heatmap(self):
        """Create engagement heatmap by phases"""
        if not self.phases.empty:
            fig = px.bar(
                self.phases,
                x='phase',
                y=['turns', 'total_words', 'questions'],
                title="Engagement by Conversation Phase",
                barmode='group'
            )
            return fig
        return None
    
    def create_speaker_metrics_radar(self):
        """Radar chart comparing speaker metrics"""
        # Normalize metrics for radar chart
        metrics_to_plot = ['turns', 'avg_words', 'questions', 'words_per_second']
        
        fig = go.Figure()
        
        for speaker in self.speaker_stats.index:
            values = []
            for metric in metrics_to_plot:
                # Normalize to 0-100 scale
                max_val = self.speaker_stats[metric].max()
                if max_val > 0:
                    normalized = (self.speaker_stats.loc[speaker, metric] / max_val) * 100
                else:
                    normalized = 0
                values.append(normalized)
            
            fig.add_trace(go.Scatterpolar(
                r=values,
                theta=metrics_to_plot,
                fill='toself',
                name=speaker
            ))
        
        fig.update_layout(
            polar=dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 100]
                )
            ),
            showlegend=True,
            title="Speaker Performance Metrics"
        )
        
        return fig
    
    def create_word_flow_chart(self):
        """Show word count flow over time"""
        fig = make_subplots(rows=2, cols=1, 
                           subplot_titles=['Word Count Over Time', 'Cumulative Word Count'])
        
        # Individual word counts
        for speaker in self.df['speaker'].unique():
            speaker_df = self.df[self.df['speaker'] == speaker]
            fig.add_trace(
                go.Scatter(
                    x=speaker_df['start_seconds'],
                    y=speaker_df['word_count'],
                    mode='markers+lines',
                    name=f'{speaker} (Individual)',
                    line=dict(width=2)
                ),
                row=1, col=1
            )
        
        # Cumulative word counts
        for speaker in self.df['speaker'].unique():
            speaker_df = self.df[self.df['speaker'] == speaker]
            cumulative_words = speaker_df['word_count'].cumsum()
            fig.add_trace(
                go.Scatter(
                    x=speaker_df['start_seconds'],
                    y=cumulative_words,
                    mode='lines',
                    name=f'{speaker} (Cumulative)',
                    line=dict(width=3)
                ),
                row=2, col=1
            )
        
        fig.update_xaxes(title_text="Time (seconds)")
        fig.update_yaxes(title_text="Word Count")
        fig.update_layout(height=600, title="Word Flow Analysis")
        
        return fig

# Create visualizations
dashboard = VisualizationDashboard(df, speaker_stats, phases, engagement)

# Display charts
dashboard.create_speaker_distribution_chart().show()
dashboard.create_timeline_chart().show()
if dashboard.create_engagement_heatmap():
    dashboard.create_engagement_heatmap().show()
dashboard.create_speaker_metrics_radar().show()
dashboard.create_word_flow_chart().show()

ValueError: String or int arguments are only possible when a DataFrame or an array is provided in the `data_frame` argument. No DataFrame was provided, but argument 'hover_data_0' is of type str or int.

## 5. Statistical Analysis

In [7]:
class StatisticalAnalyzer:
    """Perform statistical analysis on conversation data"""
    
    def __init__(self, df):
        self.df = df
    
    def basic_statistics(self):
        """Calculate basic statistical measures"""
        stats = {
            'conversation_length': {
                'total_seconds': self.df['end_seconds'].max(),
                'total_minutes': self.df['end_seconds'].max() / 60,
                'segments': len(self.df)
            },
            'word_statistics': {
                'total_words': self.df['word_count'].sum(),
                'avg_words_per_segment': self.df['word_count'].mean(),
                'median_words_per_segment': self.df['word_count'].median(),
                'std_words_per_segment': self.df['word_count'].std(),
                'words_per_minute': self.df['word_count'].sum() / (self.df['end_seconds'].max() / 60)
            },
            'timing_statistics': {
                'avg_segment_duration': self.df['duration'].mean(),
                'median_segment_duration': self.df['duration'].median(),
                'longest_segment': self.df['duration'].max(),
                'shortest_segment': self.df['duration'].min()
            }
        }
        return stats
    
    def speaker_comparison_stats(self):
        """Statistical comparison between speakers"""
        from scipy import stats as scipy_stats
        
        speakers = self.df['speaker'].unique()
        if len(speakers) < 2:
            return {"error": "Need at least 2 speakers for comparison"}
        
        comparisons = {}
        
        for i, speaker1 in enumerate(speakers):
            for speaker2 in speakers[i+1:]:
                data1 = self.df[self.df['speaker'] == speaker1]['word_count']
                data2 = self.df[self.df['speaker'] == speaker2]['word_count']
                
                if len(data1) > 1 and len(data2) > 1:
                    # T-test for word count differences
                    t_stat, p_value = scipy_stats.ttest_ind(data1, data2)
                    
                    comparisons[f"{speaker1}_vs_{speaker2}"] = {
                        'mean_words_speaker1': data1.mean(),
                        'mean_words_speaker2': data2.mean(),
                        't_statistic': t_stat,
                        'p_value': p_value,
                        'significant_difference': p_value < 0.05
                    }
        
        return comparisons
    
    def correlation_analysis(self):
        """Analyze correlations between different metrics"""
        correlation_data = self.df[['duration', 'word_count', 'char_count', 'question_marks', 'exclamation_marks']]
        correlation_matrix = correlation_data.corr()
        
        return correlation_matrix
    
    def outlier_detection(self):
        """Detect outliers in conversation segments"""
        metrics = ['duration', 'word_count', 'char_count']
        outliers = {}
        
        for metric in metrics:
            Q1 = self.df[metric].quantile(0.25)
            Q3 = self.df[metric].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_indices = self.df[
                (self.df[metric] < lower_bound) | (self.df[metric] > upper_bound)
            ].index.tolist()
            
            outliers[metric] = {
                'count': len(outlier_indices),
                'percentage': (len(outlier_indices) / len(self.df)) * 100,
                'indices': outlier_indices,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound
            }
        
        return outliers

# Perform statistical analysis
stat_analyzer = StatisticalAnalyzer(df)

basic_stats = stat_analyzer.basic_statistics()
speaker_comparisons = stat_analyzer.speaker_comparison_stats()
correlations = stat_analyzer.correlation_analysis()
outliers = stat_analyzer.outlier_detection()

print("BASIC STATISTICS:")
for category, stats in basic_stats.items():
    print(f"\n{category.upper()}:")
    for key, value in stats.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")

print("\nCORRELATION MATRIX:")
display(correlations.round(3))

print("\nOUTLIER DETECTION:")
for metric, outlier_info in outliers.items():
    print(f"{metric}: {outlier_info['count']} outliers ({outlier_info['percentage']:.1f}%)")

BASIC STATISTICS:

CONVERSATION_LENGTH:
  total_seconds: 1520.00
  total_minutes: 25.33
  segments: 67

WORD_STATISTICS:
  total_words: 1195
  avg_words_per_segment: 17.84
  median_words_per_segment: 19.00
  std_words_per_segment: 9.27
  words_per_minute: 47.17

TIMING_STATISTICS:
  avg_segment_duration: 22.19
  median_segment_duration: 23.50
  longest_segment: 29.50
  shortest_segment: 8.50

CORRELATION MATRIX:


,duration,word_count,char_count,question_marks,exclamation_marks
duration,1.000,0.215,0.244,0.233,NaN
word_count,0.215,1.000,0.975,0.129,NaN
char_count,0.244,0.975,1.000,0.112,NaN
question_marks,0.233,0.129,0.112,1.000,NaN
exclamation_marks,NaN,NaN,NaN,NaN,NaN



OUTLIER DETECTION:
duration: 2 outliers (3.0%)
word_count: 0 outliers (0.0%)
char_count: 0 outliers (0.0%)


## 6. Conversation Quality Scoring

In [10]:
class ConversationQualityScorer:
    """Score conversation quality based on quantitative metrics"""
    
    def __init__(self, df, speaker_stats, engagement_metrics, turn_metrics):
        self.df = df
        self.speaker_stats = speaker_stats
        self.engagement = engagement_metrics
        self.turn_metrics = turn_metrics
    
    def calculate_balance_score(self):
        """Score based on conversation balance (0-100)"""
        return min(100, self.engagement['conversation_balance'])
    
    def calculate_interaction_score(self):
        """Score based on interaction frequency (0-100)"""
        # Ideal interaction density is 2-4 speaker changes per minute
        ideal_density = 3
        actual_density = self.engagement['interaction_density']
        
        if actual_density <= ideal_density:
            score = (actual_density / ideal_density) * 100
        else:
            # Penalize too high frequency (might indicate interruptions/confusion)
            excess = actual_density - ideal_density
            score = 100 - (excess * 10)
        
        return max(0, min(100, score))
    
    def calculate_responsiveness_score(self):
        """Score based on response times (0-100)"""
        avg_response = self.turn_metrics['avg_response_time']
        
        if avg_response <= 2:  # Quick responses
            return 100
        elif avg_response <= 5:  # Normal responses
            return 80
        elif avg_response <= 10:  # Slower responses
            return 60
        else:  # Very slow responses
            return 30
    
    def calculate_engagement_score(self):
        """Score based on questions and engagement indicators (0-100)"""
        questions_per_segment = self.df['question_marks'].sum() / len(self.df)
        
        # Ideal range: 0.1-0.3 questions per segment
        if 0.1 <= questions_per_segment <= 0.3:
            return 100
        elif questions_per_segment < 0.1:
            return questions_per_segment / 0.1 * 100
        else:
            # Too many questions might indicate confusion
            return max(50, 100 - (questions_per_segment - 0.3) * 200)
    
    def calculate_consistency_score(self):
        """Score based on consistent participation (0-100)"""
        # Calculate coefficient of variation for word counts per speaker
        speaker_variations = []
        
        for speaker in self.df['speaker'].unique():
            speaker_words = self.df[self.df['speaker'] == speaker]['word_count']
            if len(speaker_words) > 1 and speaker_words.mean() > 0:
                cv = speaker_words.std() / speaker_words.mean()
                speaker_variations.append(cv)
        
        if speaker_variations:
            avg_variation = np.mean(speaker_variations)
            # Lower variation = higher consistency score
            return max(0, min(100, 100 - (avg_variation * 50)))
        return 50
    
    def calculate_overall_score(self):
        """Calculate weighted overall quality score"""
        scores = {
            'balance': self.calculate_balance_score(),
            'interaction': self.calculate_interaction_score(),
            'responsiveness': self.calculate_responsiveness_score(),
            'engagement': self.calculate_engagement_score(),
            'consistency': self.calculate_consistency_score()
        }
        
        # Weighted average
        weights = {
            'balance': 0.25,
            'interaction': 0.20,
            'responsiveness': 0.20,
            'engagement': 0.20,
            'consistency': 0.15
        }
        
        overall = sum(scores[key] * weights[key] for key in scores.keys())
        
        return {
            'individual_scores': scores,
            'overall_score': round(overall, 1),
            'grade': self.get_grade(overall)
        }
    
    def get_grade(self, score):
        """Convert numeric score to letter grade"""
        if score >= 90: return 'A'
        elif score >= 80: return 'B'
        elif score >= 70: return 'C'
        elif score >= 60: return 'D'
        else: return 'F'

# Calculate quality scores
scorer = ConversationQualityScorer(df, speaker_stats, engagement, turn_metrics)
quality_scores = scorer.calculate_overall_score()

print("CONVERSATION QUALITY ANALYSIS:")
print(f"Overall Score: {quality_scores['overall_score']}/100 (Grade: {quality_scores['grade']})")
print("\nIndividual Scores:")
for category, score in quality_scores['individual_scores'].items():
    print(f"  {category.capitalize()}: {score:.1f}/100")

# Create quality score visualization
fig = go.Figure()

categories = list(quality_scores['individual_scores'].keys())
scores = list(quality_scores['individual_scores'].values())

fig.add_trace(go.Scatterpolar(
    r=scores,
    theta=categories,
    fill='toself',
    name='Quality Scores'
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    showlegend=True,
    title=f"Conversation Quality Assessment (Overall: {quality_scores['overall_score']}/100)"
)

fig.write_html("plot11.html")

CONVERSATION QUALITY ANALYSIS:
Overall Score: 88.6/100 (Grade: B)

Individual Scores:
  Balance: 100.0/100
  Interaction: 88.2/100
  Responsiveness: 100.0/100
  Engagement: 73.4/100
  Consistency: 75.2/100


## 7. Export and Summary Report

In [11]:
class ReportGenerator:
    """Generate comprehensive analysis report"""
    
    def __init__(self, df, speaker_stats, phases, engagement, quality_scores, basic_stats):
        self.df = df
        self.speaker_stats = speaker_stats
        self.phases = phases
        self.engagement = engagement
        self.quality_scores = quality_scores
        self.basic_stats = basic_stats
    
    def generate_summary_table(self):
        """Generate executive summary table"""
        summary_data = {
            'Metric': [
                'Total Duration (minutes)',
                'Total Segments',
                'Unique Speakers',
                'Total Words',
                'Words per Minute',
                'Speaker Changes',
                'Avg Response Time (sec)',
                'Conversation Balance',
                'Overall Quality Score',
                'Quality Grade'
            ],
            'Value': [
                f"{self.basic_stats['conversation_length']['total_minutes']:.1f}",
                str(self.basic_stats['conversation_length']['segments']),
                str(self.df['speaker'].nunique()),
                str(self.basic_stats['word_statistics']['total_words']),
                f"{self.basic_stats['word_statistics']['words_per_minute']:.1f}",
                str(turn_metrics['speaker_changes']),
                f"{turn_metrics['avg_response_time']:.1f}",
                f"{self.engagement['conversation_balance']:.1f}%",
                f"{self.quality_scores['overall_score']}/100",
                self.quality_scores['grade']
            ]
        }
        
        return pd.DataFrame(summary_data)
    
    def export_to_csv(self, filename_prefix='transcript_analysis'):
        """Export all analysis results to CSV files"""
        # Export main data
        self.df.to_csv(f'{filename_prefix}_conversation_data.csv', index=False)
        
        # Export speaker statistics
        self.speaker_stats.to_csv(f'{filename_prefix}_speaker_stats.csv')
        
        # Export phases analysis
        if not self.phases.empty:
            self.phases.to_csv(f'{filename_prefix}_phases.csv', index=False)
        
        # Export summary
        summary = self.generate_summary_table()
        summary.to_csv(f'{filename_prefix}_summary.csv', index=False)
        
        return f"Exported: {filename_prefix}_[conversation_data|speaker_stats|phases|summary].csv"
    
    def generate_insights(self):
        """Generate key insights from the analysis"""
        insights = []
        
        # Dominance insights
        dominant_speaker = self.speaker_stats['total_words'].idxmax()
        dominance_pct = (self.speaker_stats.loc[dominant_speaker, 'total_words'] / 
                        self.speaker_stats['total_words'].sum()) * 100
        insights.append(f"Speaker '{dominant_speaker}' dominated with {dominance_pct:.1f}% of total words")
        
        # Balance insights
        if self.engagement['conversation_balance'] > 80:
            insights.append("Conversation was well-balanced between participants")
        elif self.engagement['conversation_balance'] < 50:
            insights.append("Conversation was imbalanced - one speaker dominated significantly")
        
        # Interaction insights
        if self.engagement['interaction_density'] > 4:
            insights.append("High interaction frequency - very dynamic conversation")
        elif self.engagement['interaction_density'] < 1:
            insights.append("Low interaction frequency - more monologue-style")
        
        # Quality insights
        if self.quality_scores['overall_score'] > 80:
            insights.append("High quality conversation with good engagement and flow")
        elif self.quality_scores['overall_score'] < 60:
            insights.append("Conversation quality could be improved")
        
        # Duration insights
        duration_min = self.basic_stats['conversation_length']['total_minutes']
        if duration_min > 60:
            insights.append("Long conversation - participants showed high engagement")
        elif duration_min < 10:
            insights.append("Brief conversation - might benefit from longer discussion")
        
        return insights

# Generate report
reporter = ReportGenerator(df, speaker_stats, phases, engagement, quality_scores, basic_stats)

summary_table = reporter.generate_summary_table()
insights = reporter.generate_insights()
export_status = reporter.export_to_csv()

print("EXECUTIVE SUMMARY:")
display(summary_table)

print("\nKEY INSIGHTS:")
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print(f"\n{export_status}")
print("\nAnalysis complete! All results exported and visualized.")

EXECUTIVE SUMMARY:


,Metric,Value
0,Total Duration (minutes),25.3
1,Total Segments,67
2,Unique Speakers,3
3,Total Words,1195
4,Words per Minute,47.2
5,Speaker Changes,67
6,Avg Response Time (sec),0.5
7,Conversation Balance,317.4%
8,Overall Quality Score,88.6/100
9,Quality Grade,B



KEY INSIGHTS:
1. Speaker 'Arjun' dominated with 57.9% of total words
2. Conversation was well-balanced between participants
3. High quality conversation with good engagement and flow

Exported: transcript_analysis_[conversation_data|speaker_stats|phases|summary].csv

Analysis complete! All results exported and visualized.
